In [ ]:
from google.colab import userdata

API_KEY = userdata.get("OPENROUTER_API_KEY")

if API_KEY:
    print("OpenRouter API key has been connected successfully!")
    print("Key:",API_KEY[:7]+"..."+API_KEY[-4:])
else:
    print("API key not found")

OpenRouter API key has been connected successfully!
Key: sk-or-v...2017


In [ ]:
import requests
from google.colab import userdata

API_KEY = userdata.get("OPENROUTER_API_KEY")

url = "https://openrouter.ai/api/v1/chat/completions"

headers = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json"
}

payload = {
    "model": "google/gemma-4-31b-it:free",
    "messages": [
        {
            "role": "user",
            "content": """
You are Khana Kya Hai?, an AI assistant for hostel students.

Today's hostel mess menu:
Aloo Paratha, Paneer Butter Masala, Rice, Dal, Kheer

Student allergy:
Milk / Dairy

Diet:
Vegetarian

Analyze the menu and tell me:
1. Which items should be avoided?
2. Which items appear suitable?
3. Suggest one simple cheap alternative if needed.

Important:
Do not claim medical certainty. Tell the user to verify ingredients with the mess.
"""
        }
    ],
    "temperature": 0.2
}

response = requests.post(
    url,
    headers=headers,
    json=payload,
    timeout=60
)

print("Status Code:", response.status_code)

if response.ok:
    result = response.json()
    print("\n===== GEMMA RESPONSE =====\n")
    print(result["choices"][0]["message"]["content"])
else:
    print("\n===== ERROR =====\n")
    print(response.text)

Status Code: 200

===== GEMMA RESPONSE =====

Namaste! I am **Khana Kya Hai?**, your hostel mess companion. 

Based on your **Dairy Allergy** and **Vegetarian** diet, here is the analysis of today's menu:

### 🚫 Items to Avoid
Since you have a milk/dairy allergy, please avoid the following:
*   **Paneer Butter Masala:** Contains paneer (cottage cheese), butter, and likely cream.
*   **Kheer:** A milk-based dessert.
*   **Aloo Paratha:** These are often cooked with butter or ghee. Even if the dough is plain, the frying medium is usually dairy-based in hostels.

### ✅ Suitable Items
These items are generally dairy-free, but please double-check:
*   **Rice:** Usually safe (unless cooked with ghee).
*   **Dal:** Usually safe (unless a "tadka" of ghee is added at the end).

### 💡 Simple & Cheap Alternative
Since most of the main course is dairy-heavy today, you might feel hungry. 
**Suggestion:** Keep a packet of **Roasted Peanuts or Roasted Chana** in your room. They are cheap, protein-ric

In [3]:
!pip install -q -U gradio requests

import json
import time
import requests
import gradio as gr
from google.colab import userdata

# =========================
# 1. API KEY
# =========================
API_KEY = userdata.get("OPENROUTER_API_KEY")

if not API_KEY:
    raise ValueError("OPENROUTER_API_KEY not found in Colab Secrets.")

# =========================
# 2. OPENROUTER CONFIG
# =========================
URL = "https://openrouter.ai/api/v1/chat/completions"
MODEL = "google/gemma-4-31b-it:free"

HEADERS = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json",
    "HTTP-Referer": "https://colab.research.google.com/",
    "X-Title": "Khana Kya Hai?"
}

# =========================
# 3. ALLERGEN MAP
# =========================
ALLERGEN_MAP = {
    "dairy": [
        "milk", "dairy", "paneer", "cheese",
        "butter", "cream", "curd", "yogurt",
        "kheer", "ghee"
    ],
    "milk": [
        "milk", "dairy", "paneer", "cheese",
        "butter", "cream", "curd", "yogurt",
        "kheer", "ghee"
    ],
    "peanut": [
        "peanut", "groundnut", "ground nut",
        "moongfali", "peanut butter"
    ],
    "egg": [
        "egg", "eggs", "anda"
    ],
    "soy": [
        "soy", "soya", "tofu"
    ],
    "gluten": [
        "wheat", "maida", "atta", "bread",
        "roti", "paratha", "naan"
    ]
}

def expand_allergies(text):
    items = [
        x.strip().lower()
        for x in text.split(",")
        if x.strip()
    ]

    expanded = set(items)

    for item in items:
        expanded.update(ALLERGEN_MAP.get(item, []))

    return expanded


# =========================
# 4. GEMMA API CALL
# =========================
def call_gemma(menu, allergies, diet, budget):

    system_prompt = (
        "You are Khana Kya Hai?, an AI assistant for hostel students.\n"
        "Analyze the hostel mess menu using the user's allergies, diet, and budget.\n"
        "Never claim medical certainty.\n"
        "Always tell the user to verify ingredients and cooking methods.\n"
        "Return ONLY valid JSON with these keys:\n"
        "{"
        "\"overall_status\":\"SAFE|CAUTION|HIGH_RISK\","
        "\"avoid_items\":[],"
        "\"suitable_items\":[],"
        "\"possible_meal\":\"\","
        "\"alternative\":\"\","
        "\"estimated_cost\":0,"
        "\"reason\":\"\""
        "}"
    )

    user_prompt = (
        f"HOSTEL MESS MENU: {menu}\n"
        f"ALLERGIES / INTOLERANCES: {allergies}\n"
        f"DIET: {diet}\n"
        f"DAILY BUDGET: {budget}"
    )

    payload = {
    "model": MODEL,
    "messages": [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ],
    "temperature": 0.2,
    "provider": {
        "allow_fallbacks": True
    }
}
    last_error = ""

    for attempt in range(3):

        try:
            response = requests.post(
                URL,
                headers=HEADERS,
                json=payload,
                timeout=60
            )

            if response.status_code == 200:
                result = response.json()
                return result["choices"][0]["message"]["content"]

            last_error = f"API {response.status_code}: {response.text}"

            if response.status_code == 429 and attempt < 2:
                time.sleep(3 * (attempt + 1))
                continue

            return "__ERROR__" + last_error

        except Exception as e:
            last_error = str(e)

            if attempt < 2:
                time.sleep(2)
                continue

            return "__ERROR__" + last_error

    return "__ERROR__" + last_error


# =========================
# 5. JSON PARSER
# =========================
def parse_json(text):

    cleaned = text.strip()

    # Remove markdown code fences if Gemma adds them
    if cleaned.startswith("```"):
        cleaned = cleaned.replace("```json", "")
        cleaned = cleaned.replace("```", "")
        cleaned = cleaned.strip()

    try:
        return json.loads(cleaned)

    except Exception:

        start = cleaned.find("{")
        end = cleaned.rfind("}")

        if start >= 0 and end > start:
            try:
                return json.loads(
                    cleaned[start:end + 1]
                )
            except Exception:
                return None

    return None


# =========================
# 6. SAFETY FILTER
# =========================
def safety_filter(data, allergy_text):

    blocked_words = expand_allergies(allergy_text)

    avoid_items = list(
        data.get("avoid_items") or []
    )

    suitable_items = (
        data.get("suitable_items") or []
    )

    final_suitable = []

    for item in suitable_items:

        item_lower = str(item).lower()

        blocked = any(
            word in item_lower
            for word in blocked_words
        )

        if blocked:

            if item not in avoid_items:
                avoid_items.append(item)

        else:
            final_suitable.append(item)

    data["suitable_items"] = final_suitable
    data["avoid_items"] = avoid_items

    if not final_suitable and avoid_items:
        data["overall_status"] = "HIGH_RISK"

    return data


# =========================
# 7. MAIN FUNCTION
# =========================
def analyze_food(menu, allergies, diet, budget):

    if not menu or not menu.strip():
        return "## ⚠️ Please enter today's hostel mess menu."

    if not allergies or not allergies.strip():
        return "## ⚠️ Please enter your allergy/intolerance."

    raw_response = call_gemma(
        menu,
        allergies,
        diet,
        budget
    )

    if raw_response.startswith("__ERROR__"):

        return (
            "## 🚨 Gemma Request Failed\n\n"
            "```text\n"
            + raw_response[9:]
            + "\n```"
        )

    data = parse_json(raw_response)

    if data is None:

        return (
            "## ⚠️ Unexpected Gemma Response\n\n"
            "```text\n"
            + raw_response
            + "\n```"
        )

    data = safety_filter(
        data,
        allergies
    )

    status = str(
        data.get(
            "overall_status",
            "CAUTION"
        )
    ).upper()

    if status == "SAFE":
        status_label = "🟢 SAFE-LOOKING OPTIONS"

    elif status == "HIGH_RISK":
        status_label = "🚨 HIGH RISK / AVOID"

    else:
        status_label = "🟡 CAUTION / VERIFY"

    avoid_items = data.get(
        "avoid_items"
    ) or []

    suitable_items = data.get(
        "suitable_items"
    ) or []

    avoid_text = "\n".join(
        f"- ❌ {item}"
        for item in avoid_items
    )

    suitable_text = "\n".join(
        f"- ✅ {item}"
        for item in suitable_items
    )

    if not avoid_text:
        avoid_text = "- None identified"

    if not suitable_text:
        suitable_text = "- None identified"

    return (
        "# 🍛 Khana Kya Hai?\n\n"
        f"## {status_label}\n\n"

        "### ❌ Items to Avoid\n"
        f"{avoid_text}\n\n"

        "### ✅ Suitable-looking Items\n"
        f"{suitable_text}\n\n"

        "### 🍽️ Recommended Meal\n"
        f"{data.get('possible_meal', 'No recommendation')}\n\n"

        "### 🥪 Cheap Alternative\n"
        f"{data.get('alternative', 'No alternative suggested')}\n\n"

        "### 💰 Estimated Cost\n"
        f"₹{data.get('estimated_cost', 0)}\n\n"

        "### 💡 Why?\n"
        f"{data.get('reason', 'No explanation provided.')}\n\n"

        "---\n"
        "⚠️ **Safety note:** This is an AI prototype, "
        "not medical advice. Always verify ingredients, "
        "cross-contact, and cooking method with the mess."
    )


# =========================
# 8. GRADIO UI
# =========================
with gr.Blocks(
    title="Khana Kya Hai?"
) as demo:

    gr.Markdown(
        "# 🍛 KHANA KYA HAI?\n"
        "### Ranchi Hostel Mess & Allergy Planner\n"
        "**Powered by Google's Gemma open-weight model**"
    )

    gr.Markdown(
        "Enter today's mess menu, allergy, diet and budget."
    )

    with gr.Row():

        with gr.Column():

            menu_input = gr.Textbox(
                label="🍽️ Today's Mess Menu",
                placeholder=(
                    "Aloo Paratha, Paneer Butter Masala, "
                    "Rice, Dal, Kheer"
                ),
                lines=4
            )

            allergy_input = gr.Textbox(
                label="⚠️ Allergies / Intolerances",
                placeholder="milk, dairy, peanut"
            )

            diet_input = gr.Dropdown(
                [
                    "Vegetarian",
                    "Non-Vegetarian",
                    "Vegan"
                ],
                value="Vegetarian",
                label="🥗 Dietary Preference"
            )

            budget_input = gr.Textbox(
                label="💰 Daily Budget",
                value="₹100"
            )

            analyze_button = gr.Button(
                "🔍 Analyze My Food",
                variant="primary"
            )

        with gr.Column():

            output = gr.Markdown(
                "### 👋 Enter your details and click **Analyze My Food**."
            )

    analyze_button.click(
        fn=analyze_food,
        inputs=[
            menu_input,
            allergy_input,
            diet_input,
            budget_input
        ],
        outputs=output
    )


# =========================
# 9. LAUNCH
# =========================
demo.launch(
    share=True,
    debug=True
)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://22f220a690eb37e5ee.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://22f220a690eb37e5ee.gradio.live
